In [1]:
# [Cell 1] 구글 드라이브 연결 및 필요 라이브러리 불러오기
from google.colab import drive
import pandas as pd
import numpy as np

# 구글 드라이브 마운트
drive.mount('/content/drive')

# 2025년 청소년건강행태조사 원시자료 불러오기
df = pd.read_sas('/content/drive/MyDrive/hdac_data/kyrbs2025.sas7bdat')
print(f"전체 원시자료 행 수: {len(df)}명")

# ----------------------------------------------------------------------

# [Cell 2] 고등학생 도메인 추출, 아침 결식 및 비만 변수 생성 (B-1)
# GRADE 4, 5, 6: 고등학교 1, 2, 3학년
df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)

# 아침식사 결식 변수 (F_BR: 1~3은 최근 7일 중 5일 이상 결식)
df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
                         np.where(df['F_BR'].notnull(), 0, np.nan))

# BMI 산출 (HT: cm, WT: kg)
df['bmi'] = df['WT'] / ((df['HT'] / 100) ** 2)

# 비만 변수 생성 (BMI 25 이상을 비만으로 분류하여 KeyError 예방)
df['obesity'] = np.where(df['bmi'] >= 25, 1,
                         np.where(df['bmi'].notnull(), 0, np.nan))

# 유효 인원 표식 (valid_sample: 고등학생 도메인 & 아침결식/BMI 유효응답자)
df['valid_sample'] = (df['domain_hs'] == 1) & df['bf_skip'].notnull() & df['bmi'].notnull()
valid_df = df[df['valid_sample'] == True].copy()

print(f"최종 유효 분석 대상자 수: {len(valid_df)}명")

# ----------------------------------------------------------------------

# [Cell 3] 복합표본 설계를 반영한 가중 백분율 및 평균 산출 (B-2)
def get_weighted_pct(data, var_name, weight_col='W'):
    grouped = data.groupby(var_name)[weight_col].sum()
    total_w = grouped.sum()
    return (grouped / total_w) * 100

print("\n=== 성별 가중 백분율 (%) ===")
print(get_weighted_pct(valid_df, 'SEX'))

print("\n=== 비만 여부 가중 백분율 (%) ===")
print(get_weighted_pct(valid_df, 'obesity'))

# BMI 가중 평균 및 표준오차(SE) 간이 계산
weighted_bmi_mean = np.average(valid_df['bmi'], weights=valid_df['W'])
print(f"\nBMI 가중 평균: {weighted_bmi_mean:.2f} kg/m²")

# ----------------------------------------------------------------------

# [Cell 4] 비가중 대 가중 아침 결식률 비교 (B-3)
unweighted_skip_rate = (valid_df['bf_skip'].sum() / len(valid_df)) * 100
weighted_skip_rate = (valid_df.groupby('bf_skip')['W'].sum()[1] / valid_df['W'].sum()) * 100

print("\n=== B-3 비가중 대 가중 비교 ===")
print(f"비가중 아침 결식률: {unweighted_skip_rate:.2f}%")
print(f"가중 아침 결식률: {weighted_skip_rate:.2f}%")

Mounted at /content/drive
전체 원시자료 행 수: 54170명
최종 유효 분석 대상자 수: 25077명

=== 성별 가중 백분율 (%) ===
SEX
1.0    51.551269
2.0    48.448731
Name: W, dtype: float64

=== 비만 여부 가중 백분율 (%) ===
obesity
0.0    79.943783
1.0    20.056217
Name: W, dtype: float64

BMI 가중 평균: 22.18 kg/m²

=== B-3 비가중 대 가중 비교 ===
비가중 아침 결식률: 45.36%
가중 아침 결식률: 45.68%


/tmp/ipykernel_846/910459220.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['domain_hs'] = df['GRADE'].isin([4, 5, 6]).astype(int)
/tmp/ipykernel_846/910459220.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['bf_skip'] = np.where(df['F_BR'].isin([1, 2, 3]), 1,
/tmp/ipykernel_846/910459220.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To 

In [2]:
# ==========================================
# 분석 대상자 선정 흐름도 단계별 인원수 출력
# ==========================================

# 1단계: 전체 조사 표본
print(f"1단계 (전체 조사 표본): N = {len(df):,}")

# 2단계: 고등학생 도메인 추출 (학년 GRADE가 고등학교 1~3학년인 경우)
# (보통 청소년건강행태조사에서 학년 변수 기준 확인)
hs_domain_df = df[df['GRADE'].isin([4, 5, 6])].copy() # 4, 5, 6학년 = 고1, 고2, 고3
excluded_middle = len(df) - len(hs_domain_df)
print(f"2단계 (고등학생 도메인 추출): n = {len(hs_domain_df):,}명 선택 (제외된 중학생 등: {excluded_middle:,}명)")

# 3단계: 아침식사 변수 확인 (결측치 및 비해당 확인)
br_missing = hs_domain_df['F_BR'].isna().sum()
print(f"3단계 (아침식사 변수 확인): 결측치/무응답 인원 = {br_missing:,}명")

# 4단계: 신장·체중 및 필수변수 결측치 및 이상치 제외
# (만약 이전 문서의 1,532명과 현재 766명 중 코드 상으로 실제로 필터링되는 정확한 숫자가 출력되도록 확인)
before_cleaning = len(hs_domain_df)
# 신장(HT), 체중(WT), 아침식사(F_BR) 등의 결측치 제거 조건
cleaned_df = hs_domain_df.dropna(subset=['HT', 'WT', 'F_BR']).copy()
excluded_hw = before_cleaning - len(cleaned_df)
print(f"4단계 (신장·체중 결측치 및 이상치 제외): 제외 인원 = {excluded_hw:,}명")

# 5단계: 최종 분석 유효 인원 및 결식군/비결식군 분류
# 결식군: 주 5회 이상 결식 (예: 0~2일), 비결식군: 3~7일
cleaned_df['skip_group'] = cleaned_df['F_BR'].apply(lambda x: 1 if x in [1, 2, 3] else 0)
skip_n = (cleaned_df['skip_group'] == 1).sum()
non_skip_n = (cleaned_df['skip_group'] == 0).sum()

print(f"5단계 (최종 분석 유효 인원): N = {len(cleaned_df):,}")
print(f"  - 아침식사 결식군 (0–2일): n = {skip_n:,}명")
print(f"  - 아침식사 비결식군 (3–7일): n = {non_skip_n:,}명")


1단계 (전체 조사 표본): N = 54,170
2단계 (고등학생 도메인 추출): n = 25,843명 선택 (제외된 중학생 등: 28,327명)
3단계 (아침식사 변수 확인): 결측치/무응답 인원 = 0명
4단계 (신장·체중 결측치 및 이상치 제외): 제외 인원 = 766명
5단계 (최종 분석 유효 인원): N = 25,077
  - 아침식사 결식군 (0–2일): n = 11,376명
  - 아침식사 비결식군 (3–7일): n = 13,701명


In [3]:
# ==========================================
# [Cell 6] 설계 기반 SE 함수(svy_mean) 및 Table 1 / B-3 계산
# ==========================================
W, H, C = 'W', 'STRATA', 'CLUSTER'

def svy_mean(y, domain):
    """행을 지우지 않고 domain(0/1) 안의 가중 평균(비율)과 설계 기반 SE를 계산한다."""
    d = (domain == 1) & y.notna()
    w = df[W] * d
    yv = y.fillna(0)
    est = (w * yv).sum() / w.sum()
    z = w * (yv - est) / w.sum()
    t = pd.DataFrame({'h': df[H], 'c': df[C], 'z': z}).groupby(['h', 'c'])['z'].sum().reset_index()
    var = sum(len(g) / (len(g) - 1) * ((g['z'] - g['z'].mean()) ** 2).sum()
              for _, g in t.groupby('h') if len(g) > 1)
    return est, var ** 0.5

# 도메인 표시변수 (0/1)
dom = (df['valid_sample'] == 1).astype(int)  # 고등학생 분석 대상 표시변수
dom_skip = ((df['valid_sample'] == 1) & (df['bf_skip'] == 1)).astype(int)
dom_noskip = ((df['valid_sample'] == 1) & (df['bf_skip'] == 0)).astype(int)

def report_cat_row(label, cond):
    y = cond.astype(float).where(cond.notna())
    items = []
    for name, d_mask in [('전체', dom), ('결식군', dom_skip), ('비결식군', dom_noskip)]:
        n_cnt = int((cond.fillna(False) & (d_mask == 1)).sum())
        p, se = svy_mean(y, d_mask)
        items.append(f"{name}: {n_cnt:,}명 ({p * 100:.2f}% [SE {se * 100:.2f}])")
    print(f"{label} -> " + " | ".join(items))

def report_cont_row(label, col):
    y = df[col]
    items = []
    for name, d_mask in [('전체', dom), ('결식군', dom_skip), ('비결식군', dom_noskip)]:
        n_cnt = int(((d_mask == 1) & y.notna()).sum())
        mean_val, se = svy_mean(y, d_mask)
        items.append(f"{name}: {n_cnt:,}명, 평균 {mean_val:.2f} (SE {se:.2f})")
    print(f"{label} -> " + " | ".join(items))

# 교수님 예시: 남학생 행 출력
y_male = (df['SEX'] == 1).astype(float).where(df['SEX'].notna())
p_male, se_male = svy_mean(y_male, dom)
print(f"남학생: {int(((df['SEX'] == 1) & (dom == 1)).sum()):,}명, {p_male * 100:.2f}% (SE {se_male * 100:.2f})")

print("\n=== Table 1 전체 항목 계산 ===")
report_cat_row("남학생", df['SEX'] == 1)
report_cat_row("여학생", df['SEX'] == 2)
report_cat_row("고1 (GRADE=4)", df['GRADE'] == 4)
report_cat_row("고2 (GRADE=5)", df['GRADE'] == 5)
report_cat_row("고3 (GRADE=6)", df['GRADE'] == 6)
report_cat_row("비만 (BMI>=25)", df['obesity'] == 1)
report_cat_row("비만 아님 (BMI<25)", df['obesity'] == 0)
report_cont_row("BMI (kg/m²)", 'bmi')

print("\n=== B-3 비가중 대 가중 비교 ===")
valid_mask = (dom == 1) & df['bf_skip'].notna()
unweighted_skip = (df.loc[valid_mask, 'bf_skip'] == 1).mean() * 100
w_skip, se_skip = svy_mean(df['bf_skip'], dom)
print(f"아침 결식률 -> 비가중: {unweighted_skip:.2f}% | 가중: {w_skip * 100:.2f}% (SE {se_skip * 100:.2f})")

valid_ob_mask = (dom == 1) & df['obesity'].notna()
unweighted_ob = (df.loc[valid_ob_mask, 'obesity'] == 1).mean() * 100
w_ob, se_ob = svy_mean(df['obesity'], dom)
print(f"비만율 -> 비가중: {unweighted_ob:.2f}% | 가중: {w_ob * 100:.2f}% (SE {se_ob * 100:.2f})")


남학생: 12,949명, 51.55% (SE 1.93)

=== Table 1 전체 항목 계산 ===
남학생 -> 전체: 12,949명 (51.55% [SE 1.93]) | 결식군: 5,697명 (50.02% [SE 1.95]) | 비결식군: 7,252명 (52.84% [SE 1.99])
여학생 -> 전체: 12,128명 (48.45% [SE 1.93]) | 결식군: 5,679명 (49.98% [SE 1.95]) | 비결식군: 6,449명 (47.16% [SE 1.99])
고1 (GRADE=4) -> 전체: 8,526명 (32.83% [SE 0.21]) | 결식군: 3,676명 (31.23% [SE 0.40]) | 비결식군: 4,850명 (34.18% [SE 0.36])
고2 (GRADE=5) -> 전체: 8,363명 (33.17% [SE 0.19]) | 결식군: 3,903명 (34.13% [SE 0.42]) | 비결식군: 4,460명 (32.36% [SE 0.36])
고3 (GRADE=6) -> 전체: 8,188명 (34.00% [SE 0.22]) | 결식군: 3,797명 (34.64% [SE 0.45]) | 비결식군: 4,391명 (33.46% [SE 0.40])
비만 (BMI>=25) -> 전체: 5,144명 (20.06% [SE 0.46]) | 결식군: 2,400명 (20.65% [SE 0.55]) | 비결식군: 2,744명 (19.56% [SE 0.53])
비만 아님 (BMI<25) -> 전체: 19,933명 (79.94% [SE 0.46]) | 결식군: 8,976명 (79.35% [SE 0.55]) | 비결식군: 10,957명 (80.44% [SE 0.53])
BMI (kg/m²) -> 전체: 25,077명, 평균 22.18 (SE 0.05) | 결식군: 11,376명, 평균 22.27 (SE 0.06) | 비결식군: 13,701명, 평균 22.11 (SE 0.06)

=== B-3 비가중 대 가중 비교 ===
아침 결식률 -> 비가중: 45.36%

In [4]:
# ==========================================

# --- B-1: 고등학생 도메인 안의 제외 사유 ---
hs = df[df['GRADE'].isin([4, 5, 6])]
print("=== B-1 제외 사유 ===")
print(f"고등학생 도메인: {len(hs):,}명 / 도메인 밖: {len(df) - len(hs):,}명")
print(f"F_BR 결측: {int(hs['F_BR'].isna().sum()):,}명")
print(f"HT만 결측: {int((hs['HT'].isna() & hs['WT'].notna()).sum()):,}명")
print(f"WT만 결측: {int((hs['WT'].isna() & hs['HT'].notna()).sum()):,}명")
print(f"HT·WT 동시 결측: {int((hs['HT'].isna() & hs['WT'].isna()).sum()):,}명")
print(f"하나라도 결측(제외 합계): {int(hs[['HT', 'WT', 'F_BR']].isna().any(axis=1).sum()):,}명")
print(f"최종 유효 인원: {int((dom == 1).sum()):,}명")

# --- B-1 / Table 1 전체 행: 결식군·비결식군 가중 비율 ---
y_skip = (df['bf_skip'] == 1).astype(float).where(df['bf_skip'].notna())
y_noskip = (df['bf_skip'] == 0).astype(float).where(df['bf_skip'].notna())
p1, se1 = svy_mean(y_skip, dom)
p0, se0 = svy_mean(y_noskip, dom)
print("\n=== Table 1 '전체' 행 ===")
print(f"전체: {int((dom == 1).sum()):,}명 (100.0%)")
print(f"결식군: {int(dom_skip.sum()):,}명 ({p1 * 100:.2f}% [SE {se1 * 100:.2f}])")
print(f"비결식군: {int(dom_noskip.sum()):,}명 ({p0 * 100:.2f}% [SE {se0 * 100:.2f}])")

# --- B-3: 비가중 대 가중 비교표 (차이는 반올림 전 값으로 계산) ---
def b3_row(label, col, value):
    notna = df[col].notna()
    cond = (df[col] == value)
    y = cond.astype(float).where(notna)
    n_cat = int((cond & (dom == 1)).sum())
    n_all = int(((dom == 1) & notna).sum())
    unw = n_cat / n_all * 100
    p, se = svy_mean(y, dom)
    print(f"{label}: 비가중 {n_cat:,} ({unw:.2f}%) | 가중 {p * 100:.2f}% (SE {se * 100:.2f}) "
          f"| 차이 {p * 100 - unw:+.2f}%p")

print("\n=== B-3 비가중 대 가중 ===")
b3_row("결식군", 'bf_skip', 1)
b3_row("남학생", 'SEX', 1)
b3_row("여학생", 'SEX', 2)
b3_row("고1", 'GRADE', 4)
b3_row("고2", 'GRADE', 5)
b3_row("고3", 'GRADE', 6)
b3_row("비비만", 'obesity', 0)
b3_row("비만", 'obesity', 1)

bmi_d = df.loc[dom == 1, 'bmi'].dropna()
unw_m = bmi_d.mean()
unw_se = bmi_d.std(ddof=1) / np.sqrt(len(bmi_d))
w_m, w_se = svy_mean(df['bmi'], dom)
print(f"BMI: 비가중 {unw_m:.2f} (SE {unw_se:.4f}) | 가중 {w_m:.2f} (SE {w_se:.4f}) "
      f"| 차이 {w_m - unw_m:+.2f} kg/m² (반올림 전 {w_m - unw_m:+.4f})")

# --- 설계 점검: 집락이 1개뿐인 층 (svy_mean이 분산 계산에서 제외하는 층) ---
clusters_per_stratum = df.groupby(H)[C].nunique()
print("\n=== 설계 점검 ===")
print(f"전체 층 수: {len(clusters_per_stratum):,}개 / 집락 1개뿐인 층: {int((clusters_per_stratum == 1).sum()):,}개")

=== B-1 제외 사유 ===
고등학생 도메인: 25,843명 / 도메인 밖: 28,327명
F_BR 결측: 0명
HT만 결측: 0명
WT만 결측: 0명
HT·WT 동시 결측: 766명
하나라도 결측(제외 합계): 766명
최종 유효 인원: 25,077명

=== Table 1 '전체' 행 ===
전체: 25,077명 (100.0%)
결식군: 11,376명 (45.68% [SE 0.49])
비결식군: 13,701명 (54.32% [SE 0.49])

=== B-3 비가중 대 가중 ===
결식군: 비가중 11,376 (45.36%) | 가중 45.68% (SE 0.49) | 차이 +0.32%p
남학생: 비가중 12,949 (51.64%) | 가중 51.55% (SE 1.93) | 차이 -0.09%p
여학생: 비가중 12,128 (48.36%) | 가중 48.45% (SE 1.93) | 차이 +0.09%p
고1: 비가중 8,526 (34.00%) | 가중 32.83% (SE 0.21) | 차이 -1.17%p
고2: 비가중 8,363 (33.35%) | 가중 33.17% (SE 0.19) | 차이 -0.18%p
고3: 비가중 8,188 (32.65%) | 가중 34.00% (SE 0.22) | 차이 +1.35%p
비비만: 비가중 19,933 (79.49%) | 가중 79.94% (SE 0.46) | 차이 +0.46%p
비만: 비가중 5,144 (20.51%) | 가중 20.06% (SE 0.46) | 차이 -0.46%p
BMI: 비가중 22.23 (SE 0.0240) | 가중 22.18 (SE 0.0521) | 차이 -0.04 kg/m² (반올림 전 -0.0445)

=== 설계 점검 ===
전체 층 수: 106개 / 집락 1개뿐인 층: 0개
